# Лабораторная 1. Введение в Apache Spark

## Подготовка среды в Google Colab

Установка зависимостей

In [1]:
!apt-get update -qq > /dev/null 2>&1
!apt-get install openjdk-11-jdk-headless -y -qq > /dev/null 2>&1
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz -O spark.tgz
!tar xf spark.tgz -C /content/ > /dev/null
!pip install -q findspark

Настройка переменных окружения

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

Инициализация Spark

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

## Загрузка данных

In [4]:
!wget -O stations.csv https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv

--2026-03-19 14:10:55--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv [following]
--2026-03-19 14:10:56--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/stations.csv
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 5647 (5.5K) [text/plain]
Saving to: ‘stations.csv’

stations.csv        100%[===================>]   5.51K  --.-KB/s    in 0s      

2026-03-19 14:10:56 (45.3 MB/s) - ‘stations.csv’ saved [5647/5647]



In [5]:
!wget -O trips.csv https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv

--2026-03-19 14:10:57--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv [following]
--2026-03-19 14:10:58--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/trips.csv
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 80208831 (76M) [text/plain]
Saving to: ‘trips.csv’

trips.csv           100%[===================>]  76.49M   475KB/s    in 83s     

2026-03-19 14:12:21 (943 KB/s) - ‘trips.csv’ saved [80208831/80208831]



## Решение задач

1. Найти велосипед с максимальным временем пробега.
2. Найти наибольшее геодезическое расстояние между станциями.
3. Найти путь велосипеда с максимальным временем пробега через станции.
4. Найти количество велосипедов в системе.
5. Найти пользователей потративших на поездки более 3 часов.

In [6]:
tripData = spark.read.option("header", True).option("inferSchema", True).option("timestampFormat", 'M/d/y H:m').csv("trips.csv")
tripData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [7]:
tripData.dropna().show(n=5)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|2013-08-29 11:29:00|  San Jose City Hall|              10|2013-08-29 11:30:00|  San Jose City Hall|            10|     26|       Subscriber|   95060|
|4299|      83|2013-08-29 12:02:00|South Van Ness at...|              66|2013-08-29 12:04:00|      Market at 10th|            67|    319|       Subscriber|   94103|
|4927|    

In [8]:
stationData = spark.read.option("header", True).option("inferSchema", True).option("timestampFormat", 'M/d/y').csv("stations.csv")
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [9]:
stationData.show(n=5)

+---+--------------------+------------------+-------------------+----------+--------+-------------------+
| id|                name|               lat|               long|dock_count|    city|  installation_date|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|        27|San Jose|2013-08-06 00:00:00|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|        15|San Jose|2013-08-05 00:00:00|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|        11|San Jose|2013-08-06 00:00:00|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|        19|San Jose|2013-08-05 00:00:00|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|        15|San Jose|2013-08-07 00:00:00|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
only showing top 5 rows



### Найти велосипед с максимальным временем пробега

Суммируем общее время поездок для каждого велосипеда

In [10]:
from pyspark.sql import functions as F

bike_total_duration = tripData.groupBy('bike_id')\
    .agg(F.sum('duration').alias('total_duration_sec'))

bike_total_duration.show()

+-------+------------------+
|bike_id|total_duration_sec|
+-------+------------------+
|    471|           1718831|
|    496|           1679568|
|    148|            332138|
|    463|           1722796|
|    540|           1752835|
|    392|           1789476|
|    623|           2037219|
|    243|            307458|
|    516|           1896751|
|     31|            407907|
|    580|           1034382|
|    137|           1529200|
|    251|           1282980|
|    451|           1695574|
|     85|           1214769|
|    458|           1647080|
|     65|            216922|
|    588|            266415|
|    255|            396395|
|     53|            226389|
+-------+------------------+
only showing top 20 rows



Находим велосипед с максимальным суммарным временем

In [11]:
max_bike = bike_total_duration.orderBy(F.col('total_duration_sec').desc()).limit(1)
max_bike.show()

+-------+------------------+
|bike_id|total_duration_sec|
+-------+------------------+
|    535|          18611693|
+-------+------------------+



### Найти наибольшее геодезическое расстояние между станциями

UDF для расчёта геодезического расстояния

In [12]:
from geopy.distance import geodesic
from pyspark.sql.types import DoubleType

def calc_geodesic_distance(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers

geodesic_udf = F.udf(calc_geodesic_distance, DoubleType())

Создаём уникальные пары станций (исключаем дубли и пары с самой собой)

Используем условие s1.id < s2.id, чтобы каждая пара была только один раз

In [13]:
station_pairs = stationData.alias('s1').crossJoin(stationData.alias('s2'))\
    .filter(F.col('s1.id') < F.col('s2.id'))

Вычисляем расстояние для каждой пары

In [14]:
station_distances = station_pairs.withColumn(
    'distance_km',
    geodesic_udf(F.col('s1.lat'), F.col('s1.long'), F.col('s2.lat'), F.col('s2.long'))
).filter(F.col('distance_km').isNotNull())

Находим максимальное расстояние

In [15]:
max_distance = station_distances.orderBy(F.col('distance_km').desc())\
    .limit(1).select(
        F.col('s1.id').alias('station_1_id'),
        F.col('s2.id').alias('station_2_id'),
        F.col('distance_km')
    )

max_distance.show()

+------------+------------+-----------------+
|station_1_id|station_2_id|      distance_km|
+------------+------------+-----------------+
|          16|          60|69.92096757764355|
+------------+------------+-----------------+



### Найти путь велосипеда с максимальным временем пробега через станции

Получаем ID велосипеда с максимальным временем

In [16]:
max_bike_id = max_bike.first()['bike_id']
max_bike_id

535

Получаем все поездки этого велосипеда, отсортированные по времени

In [17]:
bike_path = tripData.filter(F.col('bike_id') == max_bike_id)\
    .orderBy('start_date')\
    .select(
        F.col('id'),
        F.col('start_station_id'),
        F.col('end_station_id'),
        F.col('start_station_name'),
        F.col('end_station_name')
    )
bike_path.show()

+-----+----------------+--------------+--------------------+--------------------+
|   id|start_station_id|end_station_id|  start_station_name|    end_station_name|
+-----+----------------+--------------+--------------------+--------------------+
| 4966|              47|            70|     Post at Kearney|San Francisco Cal...|
| 5067|              70|            69|San Francisco Cal...|San Francisco Cal...|
| 5179|              69|            77|San Francisco Cal...|   Market at Sansome|
| 5199|              77|            64|   Market at Sansome|   2nd at South Park|
| 7806|              61|            42|     2nd at Townsend|    Davis at Jackson|
|11422|              58|            72|San Francisco Cit...|Civic Center BART...|
|12245|              72|            47|Civic Center BART...|     Post at Kearney|
|12485|              47|            60|     Post at Kearney|Embarcadero at Sa...|
|12558|              60|            46|Embarcadero at Sa...|Washington at Kea...|
|13107|         

### Найти количество велосипедов в системе

In [18]:
bike_count = tripData.select('bike_id').distinct().count()
bike_count

700

### Найти пользователей потративших на поездки более 3 часов

In [19]:
users_over_3h = tripData.groupBy('zip_code')\
    .agg((F.sum('duration') / 3600).alias('total_hours'))\
    .filter(F.col('total_hours') > 3)\
    .orderBy(F.col('total_hours').desc())
users_over_3h.show()

+--------+------------------+
|zip_code|       total_hours|
+--------+------------------+
|   94107|13821.433888888889|
|     nil|12701.541666666666|
|    NULL| 7700.909166666666|
|   94105| 7110.035555555555|
|   94133| 6010.465277777777|
|   94102| 5313.339166666667|
|   94103| 5313.163333333333|
|   95531| 4797.333333333333|
|   94111|3956.9436111111113|
|   95112|3539.5472222222224|
|   94109| 3349.202222222222|
|   94040|2168.8683333333333|
|   94110| 2061.648888888889|
|   94117|1917.0313888888888|
|   94301|1830.6605555555554|
|   94041|1743.4122222222222|
|   94158|1735.6019444444444|
|   94306|1541.8452777777777|
|   94025|1438.3991666666666|
|   94108|1424.3227777777777|
+--------+------------------+
only showing top 20 rows

